In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip -q install transformers==4.44.2 sentencepiece accelerate bert-score pandas numpy tqdm

In [ ]:
#make sure shared folder is in the MyDrive folder!
CURRENT_DIR = '/content/drive/MyDrive/NLP_Project'
os.chdir(CURRENT_DIR)

In [ ]:
CSV_PATH = "mistral-deepseek-250.csv"

df = pd.read_csv(CSV_PATH)
print("Rows:", len(df))
print("Columns:", list(df.columns))
df.head(3)

In [ ]:
ID_COL = "paper_id"
TEXT_COL = "paper_text"
SUM_MISTRAL_COL = "summary_mistral"
SUM_DEEPSEEK_COL = "summary_deepseek"

missing = [c for c in [ID_COL, TEXT_COL, SUM_MISTRAL_COL, SUM_DEEPSEEK_COL] if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}\nAvailable: {list(df.columns)}")

# Coerce to strings
for c in [ID_COL, TEXT_COL, SUM_MISTRAL_COL, SUM_DEEPSEEK_COL]:
    df[c] = df[c].astype(str)

In [ ]:
THINK_BLOCK_RE = re.compile(r"<think>.*?</think>", flags=re.DOTALL | re.IGNORECASE)

def clean_summary_text(text: str) -> str:
    t = "" if text is None else str(text)

    # Remove <think>...</think> blocks + stray tags
    t = THINK_BLOCK_RE.sub("", t)
    t = t.replace("<think>", "").replace("</think>", "")

    # If model included wrapper sections, remove them but keep the actual summary content
    # If "Paper:" exists, keep content after it
    if "Paper:" in t:
        t = t.split("Paper:", 1)[-1]

    # If "Instruction:" still remains, drop everything up through it
    if "Instruction:" in t:
        t = t.split("Instruction:", 1)[-1]

    # ollapse crazy whitespace, keep content
    t = re.sub(r"\n{3,}", "\n\n", t).strip()

    return t

def clean_summary_col_inplace(df: pd.DataFrame, col: str) -> None:
    df[col] = df[col].apply(clean_summary_text)

def drop_empty_summaries(df: pd.DataFrame, summary_cols: list[str]) -> pd.DataFrame:
    for c in summary_cols:
        df[c] = df[c].fillna("").astype(str).str.strip()

    mask_nonempty_all = np.ones(len(df), dtype=bool)
    for c in summary_cols:
        mask_nonempty_all &= df[c].str.len().gt(0)

    df_clean = df.loc[mask_nonempty_all].reset_index(drop=True)
    print(f"Dropped {len(df) - len(df_clean)} rows with empty summaries in {summary_cols}.")
    return df_clean

In [ ]:
# ====== CONFIG ======
TEXT_COL = "paper_text"
SUM_MISTRAL_COL = "summary_mistral"
SUM_DEEPSEEK_COL = "summary_deepseek"

SUMMARY_COLS = [SUM_MISTRAL_COL, SUM_DEEPSEEK_COL]

# QAGS-ish knobs (budget-safe but decent)
K_QUESTIONS = 3
MAX_SAMPLES = None   # None = run all rows (e.g., 250). Set 25 to test quickly.
SAVE_PATH = "qags_results.csv"

In [ ]:
print("Columns:", list(df.columns))

missing = [c for c in [TEXT_COL] + SUMMARY_COLS if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}\nAvailable: {list(df.columns)}")

# Clean in place
for c in SUMMARY_COLS:
    clean_summary_col_inplace(df, c)

# Ensure source text is string
df[TEXT_COL] = df[TEXT_COL].fillna("").astype(str)

# Drop rows where any required summary col is empty after cleaning
df = drop_empty_summaries(df, SUMMARY_COLS)

print("Rows after cleaning:", len(df))
print("\nExample cleaned Mistral:\n", df[SUM_MISTRAL_COL].iloc[0][:800])
print("\nExample cleaned DeepSeek:\n", df[SUM_DEEPSEEK_COL].iloc[0][:800])


In [ ]:
!pip install -q \
  "transformers>=4.46.0" \
  accelerate \
  pandas \
  tqdm \
  google-generativeai

In [ ]:
from transformers import pipeline
from bert_score import score as bertscore
import numpy as np
import pandas as pd
import torch
import re
from tqdm.auto import tqdm

device_id = 0 if torch.cuda.is_available() else -1
print("Using device:", "cuda" if device_id == 0 else "cpu")

# Question Generation model (same for all summaries)
qg = pipeline(
    "text2text-generation",
    model="google/flan-t5-small",
    device=device_id
)

# QA model (same for all answers)
qa = pipeline(
    "question-answering",
    model="deepset/roberta-base-squad2",
    tokenizer="deepset/roberta-base-squad2",
    device=device_id
)


In [ ]:
import numpy as np
from bert_score import score as bertscore

def clamp01(x: float) -> float:
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return np.nan
    return float(max(0.0, min(1.0, x)))

In [ ]:
MAX_SOURCE_CHARS = 16000
MAX_SUMMARY_CHARS = 4500

def clip_for_qa(text: str, max_chars: int) -> str:
    t = "" if text is None else str(text)
    return t[:max_chars]

def generate_questions_from_summary(summary_text: str, k: int = K_QUESTIONS) -> list[str]:
    s = clip_text_for_qags(summary_text, MAX_SUMMARY_CHARS)

    prompt = (
        f"Write exactly {k} short factual questions (no answers) that test the key claims in this medical research summary.\n"
        f"Return each question on a new line.\n\nSUMMARY:\n{s}"
    )
    out = qg(prompt, max_new_tokens=128, do_sample=False)[0]["generated_text"]

    qs = []
    for line in out.splitlines():
        line = line.strip(" -•\t").strip()
        if not line:
            continue
        if not line.endswith("?"):
            line += "?"
        qs.append(line)

    # de-dup, keep first k
    uniq = []
    seen = set()
    for q in qs:
        if q not in seen:
            uniq.append(q)
            seen.add(q)
    return uniq[:k]

def answer_question(context: str, question: str) -> str:
    ctx = clip_text_for_qags(context, MAX_SOURCE_CHARS)
    try:
        res = qa(question=question, context=ctx, handle_impossible_answer=True, max_answer_len=48)
        ans = (res.get("answer") or "").strip()
        return ans
    except Exception:
        return ""

def qags_score_for_pair(source_text: str, summary_text: str, k: int = K_QUESTIONS):
    src = clip_text_for_qags(source_text, MAX_SOURCE_CHARS)
    summ = clip_text_for_qags(summary_text, MAX_SUMMARY_CHARS)

    questions = generate_questions_from_summary(summ, k=k)
    if not questions:
        return np.nan, "no_questions", 0

    src_answers, sum_answers = [], []
    empty_src = 0
    empty_sum = 0

    for q in questions:
        a_src = answer_question(src, q)
        a_sum = answer_question(summ, q)

        if not a_src: empty_src += 1
        if not a_sum: empty_sum += 1

        if a_src and a_sum:
            src_answers.append(a_src)
            sum_answers.append(a_sum)

    if len(src_answers) == 0:
        return np.nan, f"no_pairs(empty_src={empty_src},empty_sum={empty_sum})", 0

    P, R, F1 = bertscore(sum_answers, src_answers, lang="en", rescale_with_baseline=True)
    score = float(F1.mean().item())

    # rescale/clamp to [0,1] for reporting (bertscore can go slightly negative after rescaling)
    score_clamped = float(np.clip(score, 0.0, 1.0))
    return score_clamped, "ok", len(src_answers)



In [ ]:
from tqdm.auto import tqdm

def compute_qags_with_diag(df_in, summary_col: str, out_prefix: str, k_questions: int = 3, max_samples=None):
    df_iter = df_in if max_samples is None else df_in.head(max_samples)

    scores = []
    valid_pairs = []
    n_questions = []
    status = []

    for _, row in tqdm(df_iter.iterrows(), total=len(df_iter), desc=f"QAGS {out_prefix}"):
        s = row.get(summary_col, "")
        src = row.get(TEXT_COL, "")

        if not isinstance(s, str) or not s.strip():
            scores.append(np.nan)
            valid_pairs.append(0)
            n_questions.append(0)
            status.append("empty_summary")
            continue

        sc, vp, nq, st = qags_score_for_pair_with_diag(src, s, k=k_questions)
        scores.append(sc)
        valid_pairs.append(vp)
        n_questions.append(nq)
        status.append(st)

    # assign back (safe even if subset)
    df_out = df_in.copy()
    df_out[f"qags_{out_prefix}"] = pd.Series(scores, index=df_iter.index)
    df_out[f"qags_valid_pairs_{out_prefix}"] = pd.Series(valid_pairs, index=df_iter.index)
    df_out[f"qags_questions_{out_prefix}"] = pd.Series(n_questions, index=df_iter.index)
    df_out[f"qags_status_{out_prefix}"] = pd.Series(status, index=df_iter.index)

    return df_out


In [ ]:
K_QUESTIONS = 3
MAX_SAMPLES = None

df_eval = df.copy()
df_eval = compute_qags_with_diag(df_eval, SUM_MISTRAL_COL, out_prefix="mistral", k_questions=K_QUESTIONS, max_samples=MAX_SAMPLES)
df_eval = compute_qags_with_diag(df_eval, SUM_DEEPSEEK_COL, out_prefix="deepseek", k_questions=K_QUESTIONS, max_samples=MAX_SAMPLES)

df_eval[[f"qags_mistral", f"qags_valid_pairs_mistral", f"qags_status_mistral"]].head()

In [ ]:
def report_qags_health(df_in, prefix: str, low_thresh: float = 0.20):
    score_col = f"qags_{prefix}"
    vp_col = f"qags_valid_pairs_{prefix}"
    st_col = f"qags_status_{prefix}"

    total = len(df_in)
    failed = df_in[score_col].isna().sum()
    ok = total - failed

    low = df_in[(df_in[st_col] == "ok") & (df_in[score_col] <= low_thresh)].shape[0]
    mean_ok = df_in.loc[df_in[st_col] == "ok", score_col].mean()

    print(f"\n[{prefix}]")
    print("  total rows:", total)
    print("  ok rows:", ok)
    print("  failed QA rows (NaN):", int(failed))
    print(f"  mean QAGS on ok rows (clamped 0-1): {mean_ok:.4f}")
    print(f"  low factuality rows (<= {low_thresh}) among ok rows:", int(low))
    print("  status breakdown:")
    print(df_in[st_col].value_counts(dropna=False).head(10))

report_qags_health(df_eval, "mistral", low_thresh=0.20)
report_qags_health(df_eval, "deepseek", low_thresh=0.20)

In [ ]:
df_eval.to_csv("qags_results_with_diag.csv", index=False)
print("Saved: qags_results_with_diag.csv")

In [ ]:
import numpy as np

def summarize_scores(series, name):
    s = series.astype(float)
    n_total = len(s)
    n_nan = int(s.isna().sum())
    n_scored = n_total - n_nan

    mean_excluding_nan = float(s.dropna().mean()) if n_scored > 0 else float("nan")
    mean_nan_as_zero = float(s.fillna(0.0).mean())

    print(f"\n{name}")
    print(f"  total rows: {n_total}")
    print(f"  scored rows (non-NaN): {n_scored}")
    print(f"  NaN rows: {n_nan}")
    print(f"  mean (exclude NaNs): {mean_excluding_nan:.4f}")
    print(f"  mean (NaNs as 0):    {mean_nan_as_zero:.4f}")

summarize_scores(df_eval["qags_mistral"], "Mistral QAGS")
summarize_scores(df_eval["qags_deepseek"], "DeepSeek QAGS")


In [ ]:
df_eval.to_csv(SAVE_PATH, index=False)
print("Saved:", SAVE_PATH)

import matplotlib.pyplot as plt

means = {
    "Mistral": float(np.nanmean(df_eval["qags_mistral"].replace(0, np.nan))),
    "DeepSeek": float(np.nanmean(df_eval["qags_deepseek"].replace(0, np.nan))),
}

plt.figure(figsize=(6,4))
plt.bar(list(means.keys()), list(means.values()))
plt.ylim(0, 1)
plt.ylabel("QAGS-style factual consistency (BERTScore F1)")
plt.title("Factual consistency of COVID side-effect summaries (QAGS-style)")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

means
